# Crypto.Net · 02 — Bitcoin

*Dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*

> **Aman dijalankan.** Notebook ini hanya membaca jaringan publik dan menandatangani secara offline; tidak pernah mem-broadcast transaksi. Kunci yang dibuat di sini adalah kunci demo — jangan pernah mengirim dana sungguhan ke sana.

Buka di VS Code dengan ekstensi **Polyglot Notebooks** (atau Jupyter dengan .NET Interactive) lalu jalankan sel secara berurutan. [English version](./02-bitcoin.en.ipynb)

Alamat (legacy, SegWit, Taproot), kunci WIF, data live dari Esplora, coin selection dan penandatanganan offline transaksi yang membelanjakan output SegWit dan Taproot.

In [ ]:
#r "nuget: Crypto.Net.Extensions, 1.0.0"

In [ ]:
using Crypto.Net.Core;
using Crypto.Net.Native;
using Crypto.Net.Wallet;
using Crypto.Net.Bitcoin;

var network = BitcoinNetwork.Testnet;
var wallet  = HdWallet.Generate();

## Alamat

BIP-44 (P2PKH), BIP-84 (P2WPKH) dan BIP-86 (P2TR, Taproot dengan tweak BIP-341).

In [ ]:
var legacy  = wallet.GetBitcoinAccount(BitcoinAddressType.LegacyP2PKH, 0, network);
var segwit  = wallet.GetBitcoinAccount(BitcoinAddressType.SegWitP2WPKH, 0, network);
var taproot = wallet.GetBitcoinAccount(BitcoinAddressType.TaprootP2TR, 0, network);

new[] { legacy, segwit, taproot }.Select(a => new
{
    a.Type,
    a.DerivationPath,
    Address = a.Address.Value,
    ScriptPubKey = Convert.ToHexStringLower(a.ScriptPubKey),
})

`BitcoinAddress.Parse` memvalidasi checksum (Base58Check, Bech32 vs Bech32m sesuai BIP-350) dan mengembalikan script output.

In [ ]:
var info = BitcoinAddress.Parse("bc1p5cyxnuxmeuwuvkwfem96lqzszd02n6xdcjrs20cac6yqjjwudpxqkedrcr");
Console.WriteLine($"{info.Type} · program {Convert.ToHexStringLower(info.Payload)}");
Console.WriteLine($"typo detected: {!BitcoinAddress.IsValid("bc1p5cyxnuxmeuwuvkwfem96lqzszd02n6xdcjrs20cac6yqjjwudpxqkedrcs")}");
Console.WriteLine($"wrong network detected: {!BitcoinAddress.IsValid(segwit.Address.Value, BitcoinNetwork.Mainnet)}");

## WIF

Wallet Import Format — yang ditampilkan `dumpprivkey` di Bitcoin Core.

In [ ]:
string wif = segwit.ExportWif();
var imported = BitcoinAccount.FromWif(wif, BitcoinAddressType.SegWitP2WPKH, network);
Console.WriteLine($"{wif[..6]}… → {imported.Address} (same: {imported.Address == segwit.Address})");

## Jaringan live (Esplora)

In [ ]:
var esplora = new EsploraClient(network);
try
{
    Console.WriteLine($"testnet tip: {await esplora.GetBlockNumberAsync():N0}");
    Console.WriteLine($"fee for ~6 blocks: {await esplora.GetFeeRateAsync(6)} sat/vB");
    Console.WriteLine($"balance of {segwit.Address}: {await esplora.GetBalanceAsync(segwit.Address)} tBTC");
}
catch (Exception e) { Console.WriteLine($"Esplora unavailable right now: {e.Message}"); }

## Membangun dan menandatangani offline

Dua UTXO tiruan — satu SegWit, satu Taproot — dibelanjakan dalam satu transaksi. Setiap input ditandatangani sesuai script-nya: ECDSA BIP-143 untuk P2WPKH, Schnorr BIP-341 untuk P2TR.

In [ ]:
var utxos = new List<BitcoinUtxo>
{
    new(new string('a', 64), 0, 60_000, segwit.ScriptPubKey),
    new(new string('b', 64), 1, 40_000, taproot.ScriptPubKey),
};

var pick = CoinSelector.Select(utxos, targetSats: 80_000, feeRateSatPerVb: 2m);
Console.WriteLine($"inputs {pick.Inputs.Count}, fee {pick.FeeSats} sats, change {pick.ChangeSats} sats");

var tx = new BitcoinTransaction();
foreach (var u in pick.Inputs) tx.AddInput(u.TxId, u.Vout);
tx.AddOutput("tb1qw508d6qejxtdg4y5r3zarvary0c5xw7kxpjzsx", 80_000, network);
if (pick.ChangeSats > 0) tx.AddOutput(segwit.ScriptPubKey, pick.ChangeSats);

// Each input is signed by the account that owns it.
for (int i = 0; i < pick.Inputs.Count; i++)
{
    var owner = pick.Inputs[i].Type == BitcoinAddressType.TaprootP2TR ? taproot : segwit;
    using var k = wallet.DerivePath(owner.DerivationPath!);
    tx.SignInput(i, k.PrivateKey.Span, pick.Inputs);
}

Console.WriteLine($"txid  {tx.TxId}");
Console.WriteLine($"size  {tx.Serialize().Length} bytes, {tx.VirtualSize} vB, weight {tx.Weight}");
Console.WriteLine(tx.ToHex());

Verifikasi tanda tangan Taproot terhadap output key yang sudah di-tweak, persis seperti yang dilakukan node.

In [ ]:
int tapIndex = pick.Inputs.ToList().FindIndex(u => u.Type == BitcoinAddressType.TaprootP2TR);
byte[] sighash   = tx.GetTaprootSigHash(tapIndex, pick.Inputs);
byte[] outputKey = taproot.ScriptPubKey[2..];
CryptoNative.SchnorrVerify(outputKey, sighash, tx.Inputs[tapIndex].Witness[0])

## Mengirim sungguhan

Dengan akun testnet yang memiliki saldo, satu panggilan mengambil UTXO, memilih koin, menambah kembalian, menandatangani dan mem-broadcast:

```csharp
TxHash id = await esplora.TransferAsync(segwit, "tb1q…", amountSats: 10_000);
var receipt = await esplora.WaitForReceiptAsync(id, timeout: TimeSpan.FromMinutes(30));
```

In [ ]:
foreach (IDisposable d in new IDisposable[] { legacy, segwit, taproot, imported, esplora, wallet }) d.Dispose();
"disposed"